## Concept 5 — Structured Output Prompts

### What is it?
Force the LLM to return a specific schema (Pydantic model, TypedDict, or JSON)
instead of free-form text. The output is directly usable in code — no parsing needed.

### Pipeline
```
Prompt → LLM (bound to schema) → Pydantic object / JSON dict → use in code
```

### Limitation (what Concept 6 fixes)
LLM can return structured data but cannot take real-world actions.
→ Concept 6 adds Tool Calling so the LLM can invoke functions/APIs.

In [ ]:
print('Hi')

In [ ]:
import sys
sys.path.insert(0, '.')

from PromptUtils import get_llm
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser, StrOutputParser
from pydantic import BaseModel, Field
from typing import List, Optional

llm = get_llm(temperature=0.0)
print('LLM ready')

### Step 1 — The Problem: Unstructured Text
Without structure, extracting data from LLM output requires fragile string parsing.

In [ ]:
unstructured_prompt = ChatPromptTemplate.from_messages([
    ('system', 'Extract the person details from the text.'),
    ('human',  '{text}'),
])
unstructured_chain = unstructured_prompt | llm | StrOutputParser()

text = 'Sarah is a 28-year-old machine learning engineer based in Bangalore.'
result = unstructured_chain.invoke({'text': text})
print(f'Unstructured output:\n{result}')
print(f'\nType: {type(result)} — hard to use in code!')

### Step 2 — Pydantic Schema with `with_structured_output`
Define the schema as a Pydantic model. The LLM always returns this exact structure.

In [ ]:
class Person(BaseModel):
    name:       str            = Field(description='Full name of the person')
    age:        int            = Field(description='Age in years')
    occupation: str            = Field(description='Job title or role')
    location:   Optional[str]  = Field(description='City or country', default=None)

# Bind schema to LLM — output is always a Person object
structured_llm = llm.with_structured_output(Person)

person_prompt = ChatPromptTemplate.from_messages([
    ('system', 'Extract the person details from the text.'),
    ('human',  '{text}'),
])
person_chain = person_prompt | structured_llm

result = person_chain.invoke({'text': text})
print(f'Structured output type: {type(result).__name__}')
print(f'Name:       {result.name}')
print(f'Age:        {result.age}')
print(f'Occupation: {result.occupation}')
print(f'Location:   {result.location}')

### Step 3 — Nested Schema
Schemas can be nested — extract complex hierarchical data.

In [ ]:
class Skill(BaseModel):
    name:  str = Field(description='Skill name')
    level: str = Field(description='Proficiency level: beginner/intermediate/expert')

class Developer(BaseModel):
    name:   str         = Field(description='Developer name')
    skills: List[Skill] = Field(description='List of technical skills with proficiency')

dev_llm   = llm.with_structured_output(Developer)
dev_chain = ChatPromptTemplate.from_messages([
    ('system', 'Extract the developer profile from the text.'),
    ('human',  '{text}'),
]) | dev_llm

dev_text = 'Ravi is an expert in Python and machine learning, has intermediate SQL skills, and is a beginner in Rust.'
dev = dev_chain.invoke({'text': dev_text})

print(f'Developer: {dev.name}')
for s in dev.skills:
    print(f'  {s.name}: {s.level}')

### Step 4 — JSON Output Parser
Alternative when you need a dict instead of a Pydantic object, or for models without `with_structured_output`.

In [ ]:
json_prompt = ChatPromptTemplate.from_messages([
    ('system', 'Extract data and return ONLY valid JSON with keys: title, author, year, summary.'),
    ('human',  '{text}'),
])
json_chain = json_prompt | llm | JsonOutputParser()

book_text = 'Sapiens was written by Yuval Noah Harari in 2011. It explores the history of humankind.'
book = json_chain.invoke({'text': book_text})
print(f'Type: {type(book)}')
print(f'Title:   {book["title"]}')
print(f'Author:  {book["author"]}')
print(f'Year:    {book["year"]}')
print(f'Summary: {book["summary"]}')

### Full Function

In [ ]:
def extract_person(text: str) -> Person:
    return person_chain.invoke({'text': text})

test_texts = [
    'Anjali is a 32-year-old data analyst working in Mumbai.',
    'Carlos, 45, is a senior software architect at a tech startup in Austin.',
]

print('Structured Person Extraction:')
for t in test_texts:
    p = extract_person(t)
    print(f'  Input: {t}')
    print(f'  → {p.name}, {p.age}, {p.occupation}, {p.location}\n')